# Practical 5 — HR Classification: Logistic Regression and KNN

## Objective
Train Logistic Regression with a 75% training split and random_state=2, report accuracy and misclassified samples, then train KNN with k=2 and random_state=0.

**How to use:** Put the required CSV file(s) in the `data/` folder, update the filename/path if needed, and run cells from top to bottom. Where a dataset is unavailable, the notebook reports that clearly rather than inventing results.

## Imports

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

DATA_DIR = Path("../data")
DATA_DIR.mkdir(exist_ok=True)

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

csv_path = DATA_DIR / "People Charm case.csv"
if not csv_path.exists():
    raise FileNotFoundError("Place 'People Charm case.csv' in the data/ folder.")
hr = pd.read_csv(csv_path)
target_candidates = ["left", "turnover", "churn", "attrition"]
target = next((c for c in target_candidates if c in hr.columns.str.lower().tolist()), None)
if target is None:
    raise KeyError("Could not identify target column. Expected one of: left, turnover, churn, attrition. Check the dataset columns.")
target = next(c for c in hr.columns if c.lower() == target)
X = hr.drop(columns=[target])
y = hr[target]
# Remove obvious identifier columns if present; IDs should not be predictive features.
id_cols = [c for c in X.columns if c.lower() in {"employeeid", "employee_id", "id"}]
X = X.drop(columns=id_cols)
numeric_cols = X.select_dtypes(include=np.number).columns.tolist()
categorical_cols = [c for c in X.columns if c not in numeric_cols]

## Logistic Regression — 75% train split, random_state=2

In [ ]:
# Fit preprocessing only on training data to avoid data leakage.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=2, stratify=y if y.nunique() > 1 else None
)
numeric_pipe = Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())])
categorical_pipe = Pipeline([("imputer", SimpleImputer(strategy="most_frequent")),
                             ("onehot", OneHotEncoder(handle_unknown="ignore"))])
preprocess = ColumnTransformer([("num", numeric_pipe, numeric_cols),
                                ("cat", categorical_pipe, categorical_cols)])
log_model = Pipeline([("preprocess", preprocess), ("model", LogisticRegression(max_iter=2000))])
log_model.fit(X_train, y_train)
log_pred = log_model.predict(X_test)
print("Logistic Regression accuracy:", round(accuracy_score(y_test, log_pred), 4))
print("Misclassified samples:", int((y_test.to_numpy() != log_pred).sum()))
print(classification_report(y_test, log_pred, zero_division=0))

## K-Nearest Neighbors — k=2, random_state=0

In [ ]:
X_train_k, X_test_k, y_train_k, y_test_k = train_test_split(
    X, y, test_size=0.25, random_state=0, stratify=y if y.nunique() > 1 else None
)
knn_preprocess = ColumnTransformer([("num", numeric_pipe, numeric_cols),
                                    ("cat", categorical_pipe, categorical_cols)])
knn_model = Pipeline([("preprocess", knn_preprocess), ("model", KNeighborsClassifier(n_neighbors=2))])
knn_model.fit(X_train_k, y_train_k)
knn_pred = knn_model.predict(X_test_k)
print("KNN (k=2) accuracy:", round(accuracy_score(y_test_k, knn_pred), 4))
print(classification_report(y_test_k, knn_pred, zero_division=0))

## Conclusion

Record the computed accuracy and misclassification count. These results depend on the exact CSV, its target column, and the preprocessing used. Stratification is applied when possible.